# Data Ingestion from ABIDE Preprocessed

This notebook handles data acquisition and preprocessing for a machine learning project using resting-state fMRI data from the ABIDE preprocessed dataset.

The goal is to construct a clean, reproducible dataset by:

* selecting a balanced subset of subjects (ASD vs control) from the phenotypic metadata  
* downloading only the required functional derivatives (ROI time series) from the public S3 bucket  
* filtering out invalid or missing entries  
* organizing the data into a consistent local directory structure  

Rather than working with raw fMRI volumes, this project uses pre-extracted regional time series (`rois_cc200`) generated by the C-PAC pipeline. Each file contains the temporal signal for ~200 brain regions, significantly reducing data size and preprocessing complexity while preserving information about inter-regional relationships.

These time series are later transformed into functional connectivity matrices by computing pairwise correlations between regions. This representation serves as a common input across multiple models, including both standard machine learning approaches and graph-based methods.

By the end of this notebook, we obtain:

* a curated list of valid subjects  
* locally stored ROI time series data  
* a clean labels file linking each subject to its diagnosis  

This setup isolates data collection from downstream modeling, ensuring that all experiments operate on a consistent and reproducible dataset.

In [1]:
!pip install --quiet numpy pandas tqdm requests 


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path
import requests
from tqdm import tqdm


In [3]:
# Load data

df = pd.read_csv("data/Phenotypic_V1_0b_preprocessed1.csv")
print("Total subjects:", len(df))

# Drop useless rows and keep only valid labels (DX_GROUP=1 for ASD, 2 for control)
df = df[df["FILE_ID"] != "no_filename"]
df = df[df["DX_GROUP"].isin([1, 2])]

print("Remaining subjects:", len(df))
print(df["DX_GROUP"].value_counts())


Total subjects: 1112
Remaining subjects: 1035
DX_GROUP
2    530
1    505
Name: count, dtype: int64


In [4]:
# Use full dataset (no downsampling)
subjects = df.copy().reset_index(drop=True)

file_ids = subjects["FILE_ID"].tolist()

print("Selected subjects:", len(file_ids))
print(subjects["DX_GROUP"].value_counts())

subjects.head()

Selected subjects: 1035
DX_GROUP
2    530
1    505
Name: count, dtype: int64


,Unnamed: 0.1,Unnamed: 0,SUB_ID,X,subject,SITE_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,...,qc_notes_rater_1,qc_anat_rater_2,qc_anat_notes_rater_2,qc_func_rater_2,qc_func_notes_rater_2,qc_anat_rater_3,qc_anat_notes_rater_3,qc_func_rater_3,qc_func_notes_rater_3,SUB_IN_SMP
0,1,2,50003,2,50003,PITT,Pitt_0050003,1,1,24.45,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1
1,2,3,50004,3,50004,PITT,Pitt_0050004,1,1,19.09,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1
2,3,4,50005,4,50005,PITT,Pitt_0050005,1,1,13.73,...,NaN,OK,NaN,maybe,ic-parietal-cerebellum,OK,NaN,OK,NaN,0
3,4,5,50006,5,50006,PITT,Pitt_0050006,1,1,13.37,...,NaN,OK,NaN,maybe,ic-parietal slight,OK,NaN,OK,NaN,1
4,5,6,50007,6,50007,PITT,Pitt_0050007,1,1,17.78,...,NaN,OK,NaN,maybe,ic-cerebellum_temporal_lob,OK,NaN,OK,NaN,1


In [5]:
# Set up directories
base_dir = Path("data") / "abide_fmri"
ts_dir = base_dir / "timeseries"

ts_dir.mkdir(parents=True, exist_ok=True)

def get_ts_url(fid):
    return f"https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/filt_global/rois_cc200/{fid}_rois_cc200.1D"


In [6]:
# Download function
def download_file(url, path):
    if path.exists():
        return True
    
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(path, "wb") as f:
                f.write(r.content)
            return True
    except:
        pass
    
    return False


In [7]:
# Download files and keep track of successful downloads (valid subjects)
valid_subjects = []

for fid in tqdm(file_ids):
    ts_path = ts_dir / f"{fid}.1D"
    
    if download_file(get_ts_url(fid), ts_path):
        valid_subjects.append(fid)

print("Downloaded:", len(valid_subjects))

subjects = subjects[subjects["FILE_ID"].isin(valid_subjects)]

cols = ["FILE_ID", "DX_GROUP", "AGE_AT_SCAN", "SEX", "SITE_ID"]
subjects_clean = subjects[cols].copy()
subjects_clean.to_csv(base_dir / "subjects_clean.csv", index=False)

print("Final usable subjects:", len(subjects_clean))

100%|██████████| 1035/1035 [31:35<00:00,  1.83s/it]

Downloaded: 1035
Final usable subjects: 1035
